In [ ]:
import numpy as np
print("All modules imported successfully!")

In [ ]:
def sigmoid(X):
    X = np.clip(X, -500, 500)
    return 1 / (1 + np.exp(-X))
def softmax(X):
    e_x = np.exp(X - np.max(X))
    return e_x / e_x.sum(axis=0)

In [ ]:
class LSTM:
    def __init__(self, input_size, hidden_size, output_size,learning_rate=0.01):
        self.is_dim = input_size
        self.hs = hidden_size
        self.lr = learning_rate

        concat_size = self.is_dim + self.hs 
        scale = 1.0 / np.sqrt(self.hs)

        self.weights = {
            'Wf': np.random.randn(self.hs, concat_size) * scale,
            'bf': np.ones((self.hs, 1)),

            'Wi': np.random.randn(self.hs, concat_size) * scale,
            'bi': np.zeros((self.hs, 1)),

            'Wc': np.random.randn(self.hs, concat_size) * scale,
            'bc': np.zeros((self.hs, 1)),


            'Wo': np.random.randn(self.hs, concat_size) * scale,
            'bo': np.zeros((self.hs, 1)),

            'Wy': np.random.randn(output_size, self.hs) * scale,
            'by': np.zeros((output_size, 1)),
        }

    def forward(self, X):
        """
        X: list or array of input vectors over time. Shape: (seq_len, input_size, 1)
        """
        seq_len = len(X)

        H = {}
        C = {}
        C_tilde = {}
        f, i, O = {}, {}, {}
        y_preds = {}

        H[-1] = np.zeros((self.hs, 1))
        C[-1] = np.zeros((self.hs, 1))

        for t in range(seq_len):
            concat_input = np.vstack((H[t-1], X[t]))
            f[t] = sigmoid(np.dot(self.weights['Wf'], concat_input) + self.weights['bf'])
            i[t] = sigmoid(np.dot(self.weights['Wi'], concat_input) + self.weights['bi'])
            C_tilde[t] = np.tanh(np.dot(self.weights['Wc'], concat_input) + self.weights['bc'])
            O[t] = sigmoid(np.dot(self.weights['Wo'], concat_input) + self.weights['bo'])

            C[t] = (f[t] * C[t-1]) + (i[t] * C_tilde[t])
            H[t] = O[t] * np.tanh(C[t])
            y_preds[t] = np.dot(self.weights['Wy'], H[t]) + self.weights['by']

        caches = (X, H, C, C_tilde, f, i, O)
        return y_preds, caches

    def backward(self, y_preds, caches, y_true):
        X, H, C, C_tilde, f, i, O = caches
        seq_len = len(X)

        grads = {k: np.zeros_like(v) for k, v in self.weights.items()}
        dh_next = np.zeros((self.hs, 1))
        dc_next = np.zeros((self.hs, 1))

        for t in reversed(range(seq_len)):
            dy = y_preds[t] - y_true[t]

            grads['Wy'] += np.dot(dy, H[t].T)
            grads['by'] += dy

            dht_local = np.dot(self.weights['Wy'].T, dy) 
            dht = dht_local + dh_next

            dct_local = dht * O[t] * (1 - np.tanh(C[t]) ** 2)
            dct = dct_local + dc_next

            concat_input = np.vstack((H[t-1], X[t]))

            dZo = (dht * np.tanh(C[t])) * (O[t] * (1 - O[t]))
            dZf = (dht * C[t-1]) * (f[t] * (1 - f[t]))
            dZi = (dht * C_tilde[t]) * (i[t] * (1 - i[t]))
            dZc = (dct * i[t]) * (1 - C_tilde[t] ** 2)

            grads['Wf'] += np.dot(dZf, concat_input.T); grads['bf'] += dZf
            grads['Wi'] += np.dot(dZi, concat_input.T); grads['bi'] += dZi
            grads['Wc'] += np.dot(dZc, concat_input.T); grads['bc'] += dZc
            grads['Wo'] += np.dot(dZo, concat_input.T); grads['bo'] += dZo

            dZ_concat = np.dot(self.weights['Wf'].T, dZf) + \
                        np.dot(self.weights['Wi'].T, dZi) + \
                        np.dot(self.weights['Wc'].T, dZc) + \
                        np.dot(self.weights['Wo'].T, dZo)
            
            dh_next = dZ_concat[:self.hs, :]
            dc_next = dct * f[t]

        for key in grads:
            grads[key] = np.clip(grads[key], -5, 5)

        return grads

    def update_params(self, grads):
        for key in self.weights.keys():
            self.weights[key] -= self.lr * grads[key]

    def train(self, dataset, epochs=100):
        for epoch in range(epochs):
            total_loss = 0
            for x_seq, y_seq in dataset:
                y_preds, caches = self.forward(x_seq)
                seq_loss = 0
                for t in range(len(x_seq)):
                    seq_loss += -np.sum(y_seq[t] * np.log(y_preds[t] + 1e-8))
                total_loss += seq_loss
                grads = self.backward(y_preds, caches, y_seq)
                self.update_params(grads)
            if (epoch + 1) % 10 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataset):.4f}')

     

    

0. Notation and Forward Pass ReferenceLet $T$ be the sequence length, $hs$ be the hidden size, and $is$ be the input size. Let $Z$ denote the pre-activation linear transformations.Forward Equations:Forget Gate: $f_t = \sigma(W_f[H_{t-1}, X_t] + b_f)$Input Gate: $i_t = \sigma(W_i[H_{t-1}, X_t] + b_i)$Candidate Cell State: $\tilde{C}_t = \tanh(W_c[H_{t-1}, X_t] + b_c)$Output Gate: $O_t = \sigma(W_o[H_{t-1}, X_t] + b_o)$Cell State: $C_t = (f_t \otimes C_{t-1}) \oplus (i_t \otimes \tilde{C}_t)$Hidden State: $H_t = O_t \otimes \tanh(C_t)$Prediction: $\hat{y}_t = \text{softmax}(W_y H_t + b_y)$(Note: $\otimes$ denotes pointwise multiplication, $\oplus$ denotes pointwise addition, and $[H_{t-1}, X_t]$ is vertical concatenation).Step 1: The Loss and Initial ErrorFor a given sequence, the total loss $L$ is the sum of losses at each time step: $L = \sum_{t=1}^T L_t$.The error enters the LSTM at time $t$ via the derivative of the loss with respect to the raw output prediction ($d\hat{y}_t$).We propagate this into the hidden state to get the Local Gradient:$$dH_t^{\text{local}} = W_y^T \cdot d\hat{y}_t$$Shape: $(hs \times 1)$Step 2: The Core Principle of BPTT (Dual-Flow State Management)Unlike feedforward networks, an LSTM cell at time $t$ receives gradients from two directions: from the output above it (local) and from the sequence ahead of it (future).We define our master accumulating gradients:Total Hidden State Error: $dH_t = dH_t^{\text{local}} + dH_{t+1}^{\text{future}}$Total Cell State Error: $dC_t = dC_{t}^{\text{local}} + dC_{t+1}^{\text{future}}$(For the final time step $T$, $dH_{T+1}^{\text{future}} = 0$ and $dC_{T+1}^{\text{future}} = 0$.)Step 3: Unpacking the Hidden State ($H_t$)We must push $dH_t$ backwards through the equation: $H_t = O_t \otimes \tanh(C_t)$.Using the chain rule, this splits into two paths:1. Gradient to the Output Gate ($dO_t$):Because $O_t$ acts as a multiplier to the cell state path, its partial derivative is simply the other term:$$dO_t = dH_t \otimes \tanh(C_t)$$2. Local Gradient to the Cell State ($dC_t$):We push $dH_t$ through the output gate multiplier and the $\tanh$ derivative ($1 - \tanh^2(x)$). Because $C_t$ also receives error from the future, we add (+=) this local error to our running $dC_t$ accumulator:$$dC_t \mathrel{+}= dH_t \otimes O_t \otimes (1 - \tanh^2(C_t))$$Step 4: Unpacking the Cell State ($C_t$)We push our fully loaded $dC_t$ backwards through the memory update equation: $C_t = (f_t \otimes C_{t-1}) \oplus (i_t \otimes \tilde{C}_t)$.Because addition routes the gradient equally to both sides, we use the chain rule on the pointwise multiplications:1. Gradient to the Forget Gate ($df_t$):The partial derivative of $(f_t \otimes C_{t-1})$ with respect to $f_t$ treats $C_{t-1}$ as a constant multiplier.$$df_t = dC_t \otimes C_{t-1}$$2. Gradient to the Input Gate ($di_t$):$$di_t = dC_t \otimes \tilde{C}_t$$3. Gradient to the Candidate Cell State ($d\tilde{C}_t$):$$d\tilde{C}_t = dC_t \otimes i_t$$4. Gradient to the Previous Cell State (The "Future" Error for $t-1$):This is the error that travels back in time along the cell state highway. The forget gate acts as the physical valve for this gradient flow.$$dC_{t-1}^{\text{future}} = dC_t \otimes f_t$$Step 5: Backwards Through the Activations ($dZ$)Before updating weights, we push the gate gradients through the derivatives of their respective activation functions (Sigmoid or Tanh) to get the pre-activation gradients ($dZ$).Derivative of Sigmoid: $\sigma(x) \times (1 - \sigma(x))$Derivative of Tanh: $1 - \tanh^2(x)$Applying these to our gate gradients:$dZ_o = dO_t \otimes O_t \otimes (1 - O_t)$$dZ_f = df_t \otimes f_t \otimes (1 - f_t)$$dZ_i = di_t \otimes i_t \otimes (1 - i_t)$$dZ_c = d\tilde{C}_t \otimes (1 - \tilde{C}_t^2)$(All $dZ$ vectors maintain the shape $hs \times 1$).Step 6: Calculating Weight and Bias GradientsThe linear transformation for any gate $g$ is $Z_g = W_g [H_{t-1}, X_t] + b_g$.To find the weight gradients, we take the outer product of the pre-activation gradient ($dZ$) and the transpose of the concatenated inputs.Note: In a loop over time $T$, we accumulate (+=) these gradients at every step.Forget Gate: $dW_f \mathrel{+}= dZ_f \cdot [H_{t-1}, X_t]^T$$db_f \mathrel{+}= dZ_f$Input Gate: $dW_i \mathrel{+}= dZ_i \cdot [H_{t-1}, X_t]^T$$db_i \mathrel{+}= dZ_i$Candidate State: $dW_c \mathrel{+}= dZ_c \cdot [H_{t-1}, X_t]^T$$db_c \mathrel{+}= dZ_c$Output Gate: $dW_o \mathrel{+}= dZ_o \cdot [H_{t-1}, X_t]^T$$db_o \mathrel{+}= dZ_o$(Shape Check: $dW$ matrices yield $hs \times (hs + is)$).Step 7: Passing Error Down and BackFinally, the error must flow out of the bottom of the LSTM cell to the inputs, and backward to the previous time step. Because the concatenated input $[H_{t-1}, X_t]$ was passed to all four gates, we sum the gradients from all four linear transformations:$$dZ_{\text{concat}} = (W_f^T \cdot dZ_f) + (W_i^T \cdot dZ_i) + (W_c^T \cdot dZ_c) + (W_o^T \cdot dZ_o)$$Shape: $((hs + is) \times 1)$We split this concatenated vector vertically to route the errors to their proper destinations:$dH_{t-1}^{\text{future}}$: The top $hs$ rows. This acts as the future hidden state gradient for step $t-1$.$dX_t$: The bottom $is$ rows. This is the error passed down to the current step's lower layers (e.g., an Embedding layer).The BPTT step is now complete. The loop continues to $t-1$.